# 交叉验证与泄漏

**面试回答：**每一折必须先切分，再在训练折拟合标准化/填补/词表，最后只 transform 验证折；同一用户应在同一折。

## 真实案例

10 笔退款订单来自 5 位用户，每位用户两笔；随机按订单切分会把用户模式泄漏到验证。

In [1]:
import numpy as np  # 导入 NumPy 手写折划分。
order=np.array(['O01','O02','O03','O04','O05','O06','O07','O08','O09','O10'])  # 构造订单编号。
user=np.array(['u1','u1','u2','u2','u3','u3','u4','u4','u5','u5'])  # 记录用户分组键。
amount=np.array([20,25,40,43,60,58,80,84,120,126],dtype=float)  # 记录订单金额。
y=np.array([0,0,0,0,1,1,1,1,1,1])  # 记录退款标签。
print('订单 | 用户 | 金额 | 退款')  # 输出原始账本表头。
for a,b,c,d in zip(order,user,amount,y):  # 逐条展示订单。
    print(a,b,c,int(d))  # 输出一条业务样本。

订单 | 用户 | 金额 | 退款
O01 u1 20.0 0
O02 u1 25.0 0
O03 u2 40.0 0
O04 u2 43.0 0
O05 u3 60.0 1
O06 u3 58.0 1
O07 u4 80.0 1
O08 u4 84.0 1
O09 u5 120.0 1
O10 u5 126.0 1


## Baseline / 基线

错误基线随机按订单切分，并以用户历史平均标签作为特征，验证用户在训练中已出现。

In [2]:
random_valid=np.array([1,3,5,7,9])  # 构造按订单随机抽取的验证索引。
random_train=np.array([i for i in range(len(order)) if i not in random_valid])  # 构造随机训练索引。
seen_rate={u:y[random_train][user[random_train]==u].mean() for u in np.unique(user[random_train])}  # 错误地从训练中读取同用户历史标签。
leaky_pred=np.array([seen_rate.get(u,.5) for u in user[random_valid]])  # 对验证订单使用泄漏用户画像。
leaky_acc=float(np.mean((leaky_pred>=.5)==y[random_valid]))  # 计算虚高随机切分准确率。
print('随机验证用户:',user[random_valid].tolist(),'泄漏准确率=',leaky_acc)  # 输出泄漏基线。

随机验证用户: ['u1', 'u2', 'u3', 'u4', 'u5'] 泄漏准确率= 1.0


In [3]:
folds=[np.where(np.isin(user,['u1','u2']))[0],np.where(np.isin(user,['u3','u4']))[0],np.where(user=='u5')[0]]  # 按用户构造三个 group folds。
scores=[]  # 保存每个安全折的分数。
for valid in folds:  # 逐个运行分组验证折。
    train=np.array([i for i in range(len(order)) if i not in valid])  # 取不含验证用户的训练订单。
    mean=amount[train].mean()  # 只在训练折拟合金额均值。
    threshold=mean  # 用训练均值构造简单金额风险规则。
    pred=(amount[valid]>=threshold).astype(int)  # 只用验证订单的可见金额预测。
    scores.append(float(np.mean(pred==y[valid])))  # 保存当前折准确率。
    print('验证用户=',np.unique(user[valid]).tolist(),'训练均值=',round(mean,1),'准确率=',scores[-1])  # 输出每折中间量。
print('Group CV 均值/标准差=',round(float(np.mean(scores)),3),round(float(np.std(scores)),3))  # 输出安全评估汇总。

验证用户= ['u1', 'u2'] 训练均值= 88.0 准确率= 1.0
验证用户= ['u3', 'u4'] 训练均值= 62.3 准确率= 0.5
验证用户= ['u5'] 训练均值= 51.2 准确率= 1.0
Group CV 均值/标准差= 0.833 0.236


## 结果解读

随机订单切分测到的是“见过同一用户后”的能力；分组交叉验证更接近新用户泛化。每折的预处理统计量不同且应被记录。

In [4]:
global_mean=amount.mean()  # 错误地在全量订单上拟合均值。
fold_mean=amount[folds[0]^0].mean() if False else amount[[i for i in range(len(order)) if i not in folds[0]]].mean()  # 计算第一折真正训练均值。
print('失败：全量均值=',round(global_mean,1))  # 输出包含验证订单的统计量。
print('修复：第一折训练均值=',round(fold_mean,1))  # 输出安全统计量。
print('生产差距：需固定切分键、时间截止点、样本哈希和最终一次性测试集。')  # 说明生产要求。

失败：全量均值= 65.6
修复：第一折训练均值= 88.0
生产差距：需固定切分键、时间截止点、样本哈希和最终一次性测试集。


In [5]:
print('安全 Group CV 折数=', len(folds), '各折样本数=', [len(fold) for fold in folds])  # 输出分组切分的可复核摘要。

安全 Group CV 折数= 3 各折样本数= [4, 4, 2]


## 失败案例与修复

将同一用户拆到两侧是泄漏；将全量均值用于每折也是泄漏。修复是 Group/Time split 后按折 fit。

In [6]:
assert len(order)>=5  # 保护样本数。
assert leaky_acc>=np.mean(scores)  # 保护泄漏评分不低于安全分组评分。
assert global_mean!=fold_mean  # 保护全量与折内统计不同。
assert all(len(np.intersect1d(user[v],user[[i for i in range(len(order)) if i not in v]]))==0 for v in folds)  # 保护用户不跨折。